In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

In [2]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

BASE_PATH = "../"
LINKING_DATASET = BASE_PATH + "data/linking/"
MODEL_SAVE_PATH = BASE_PATH + "models/e5-linker/"
TUNED_MODEL_SAVE_PATH = BASE_PATH + "models/e5-linker-tuned/"

In [9]:
import os
import glob
import pandas as pd
from PIL import Image
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

In [10]:
all_csv_files = glob.glob(LINKING_DATASET + "*.csv")
df = pd.concat([pd.read_csv(f) for f in all_csv_files], ignore_index=True)
df = df.dropna(subset=['spoken_text'])

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

In [11]:
def prepare_dataset(df):
    queries = ["query: " + str(x) for x in df["spoken_text"]]
    passages = ["passage: " + str(x) for x in df["context_content"]]

    return Dataset.from_dict({
        "anchor": queries,
        "positive": passages,
    })


In [17]:
model_name = "intfloat/multilingual-e5-base"
model = SentenceTransformer(model_name, device=device, token=HF_TOKEN)

In [23]:
train_dataset = prepare_dataset(train_df)
val_dataset = prepare_dataset(val_df)

loss = MultipleNegativesRankingLoss(model)

In [24]:
training_args = SentenceTransformerTrainingArguments(
    output_dir=MODEL_SAVE_PATH,
    eval_strategy="epoch", 
    save_strategy="epoch",

    save_total_limit=1,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    warmup_steps=100,
    fp16=True,
    logging_steps=10,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=loss,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.093700,0.180387
2,0.122400,0.178755
3,0.022100,0.152008
4,0.141400,0.174699
5,0.061500,0.197749
6,0.059600,0.172480
7,0.079400,0.192288
8,0.038400,0.189246
9,0.050200,0.174267
10,0.042800,0.178207


TrainOutput(global_step=1180, training_loss=0.06486864116484836, metrics={'train_runtime': 294.0161, 'train_samples_per_second': 63.84, 'train_steps_per_second': 4.013, 'total_flos': 0.0, 'train_loss': 0.06486864116484836, 'epoch': 10.0})

In [ ]:
model.save_pretrained(TUNED_MODEL_SAVE_PATH)

: 